In [28]:
# ====================== CLASS EXERCISE (ONE-CELL) ======================
# Goal: Fine-tune BERT (bert-base-uncased) on Kaggle IMDB (binary sentiment)
# Tooling: kagglehub (to download dataset), Hugging Face Transformers, scikit-learn
# Instructions: Complete each TODO following the high-level hints.
# Tip: Runtime -> Change runtime type -> GPU

# --- (Optional) Installs: uncomment if needed ---
%pip install -U kagglehub transformers scikit-learn pandas

import os, glob
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)

# Avoid W&B login prompts
os.environ["WANDB_DISABLED"] = "true"

# -----------------------------
# 0) Constants & hyperparams
# -----------------------------
DATASET_ID = "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 256
N_TRAIN, N_VAL, N_TEST = 8000, 2000, 2000   # you may increase for better results
LR = 2e-5
EPOCHS = 2
BATCH_TRAIN = 16
BATCH_EVAL  = 32

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Torch: 2.8.0+cu126 | CUDA: True


# Descarga del dataset de Kaggle
- Utilizamos la librería `kagglehub`.

In [29]:
# -------------------------------------------------------
# 1) Download IMDB from Kaggle with kagglehub  (TODO)
# -------------------------------------------------------
# TODO: Use kagglehub to download the dataset and get the local path.
# Hint:
#   import kagglehub
#   data_root = kagglehub.dataset_download(DATASET_ID)
#   Find the CSV file (e.g., with glob) and set csv_path pointing to it.

# --- YOUR CODE HERE ---
import kagglehub
data_root = kagglehub.dataset_download(DATASET_ID)
csv_candidates = glob.glob(os.path.join(data_root, "**", "*.csv"), recursive=True)
csv_path = csv_candidates[0]
print("CSV:", csv_path)
# ----------------------

# Tiny safety check (remove if you prefer)
if not os.path.exists(csv_path):
    raise FileNotFoundError("CSV not found. Check your kagglehub download code.")

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
CSV: /kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


# Cargamos el dataset
- Utilizamos `pandas`.

In [30]:
# 2) Load & prepare dataframe (binary labels)  (TODO)
# -------------------------------------------------------
# TODO: Read the CSV into a pandas DataFrame; expected columns: 'review', 'sentiment'.
# Map sentiment to integers: negative -> 0, positive -> 1
# Rename 'review' -> 'text', keep only ['text', 'label']

# --- YOUR CODE HERE ---
df = pd.read_csv(csv_path)
df["sentiment"] = df["sentiment"].map({"positive": 1, "negative": 0})
df = df.rename(columns={"review": "text", "sentiment": "label"})[["text", "label"]]
print(df.head(10))

                                                text  label
0  One of the other reviewers has mentioned that ...      1
1  A wonderful little production. <br /><br />The...      1
2  I thought this was a wonderful way to spend ti...      1
3  Basically there's a family where a little boy ...      0
4  Petter Mattei's "Love in the Time of Money" is...      1
5  Probably my all-time favorite movie, a story o...      1
6  I sure would like to see a resurrection of a u...      1
7  This show was an amazing, fresh & innovative i...      0
8  Encouraged by the positive comments about this...      0
9  If you like original gut wrenching laughter yo...      1


In [31]:
# 3) Train/val/test split & optional subsampling  (TODO)
# -------------------------------------------------------
# TODO: Make a stratified split: 80% train, 10% val, 10% test (two-step split is fine).
# Then (optionally) subsample to N_TRAIN / N_VAL / N_TEST for faster training.

# --- YOUR CODE HERE ---
df_train, df_temp = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
df_val, df_test = train_test_split(df_temp, test_size=0.5, stratify=df_temp["label"], random_state=42)
df_train = df_train.sample(n=min(N_TRAIN, len(df_train)), random_state=42)
df_val   = df_val.sample(n=min(N_VAL,   len(df_val)),   random_state=42)
df_test  = df_test.sample(n=min(N_TEST,  len(df_test)),  random_state=42)
print(len(df_train), len(df_val), len(df_test))

8000 2000 2000


In [32]:
# 4) Tokenizer, datasets & collator  (TODO)
# -------------------------------------------------------
# TODO: Load tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME).
# Create a torch.utils.data.Dataset that returns dicts with keys:
#   input_ids, attention_mask, labels
# Hint: tokenize with truncation=True, padding=False; then use DataCollatorWithPadding.

# --- YOUR CODE HERE ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_batch(texts):
    return tokenizer(texts, truncation=True, padding=False, max_length=MAX_LEN)

class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        # tokenize texts and build list of dicts
        self.encodings = tokenizer(list(texts), truncation=True, padding=False, max_length=MAX_LEN)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_ds = IMDBDataset(df_train["text"], df_train["label"], tokenizer)
val_ds   = IMDBDataset(df_val["text"],   df_val["label"],   tokenizer)
test_ds  = IMDBDataset(df_test["text"],  df_test["label"],  tokenizer)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [33]:
# 5) Model & metrics  (TODO)
# -------------------------------------------------------
# TODO: Load AutoModelForSequenceClassification with num_labels=2 (id2label/label2id optional).
# Implement compute_metrics returning accuracy and macro_f1 (use sklearn).

# --- YOUR CODE HERE ---
id2label = {0:"negative", 1:"positive"}
label2id = {"negative":0, "positive":1}
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro")}

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
# 6) TrainingArguments & Trainer  (TODO)
# -------------------------------------------------------
# TODO: Fill TrainingArguments with:
#   output_dir="out-bert-imdb", learning_rate=LR,
#   per_device_train_batch_size=BATCH_TRAIN, per_device_eval_batch_size=BATCH_EVAL,
#   num_train_epochs=EPOCHS, weight_decay=0.01,
#   fp16=torch.cuda.is_available(), evaluation_strategy="epoch",
#   save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="macro_f1",
#   logging_steps=50, report_to="none"
#
# Then build the Trainer with (model, args, train_ds, val_ds, collator, tokenizer, compute_metrics).

# --- YOUR CODE HERE ---
args = TrainingArguments(
    output_dir="out-bert-imdb", learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN, per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS, weight_decay=0.01,
    fp16=torch.cuda.is_available(), eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    logging_steps=50, report_to="none"
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator, processing_class=tokenizer, compute_metrics=compute_metrics)

In [35]:
# 7) Train & evaluate  (TODO)
# -------------------------------------------------------
# TODO: Call trainer.train(), then evaluate on val and test. Print the metrics.

# --- YOUR CODE HERE ---
trainer.train()
print("VALID:", trainer.evaluate(val_ds))
print("TEST :",  trainer.evaluate(test_ds))

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.252600,0.255610,0.904000,0.903908
2,0.098700,0.324727,0.900500,0.900491


VALID: {'eval_loss': 0.25560998916625977, 'eval_accuracy': 0.904, 'eval_macro_f1': 0.9039076552567017, 'eval_runtime': 7.0063, 'eval_samples_per_second': 285.458, 'eval_steps_per_second': 8.992, 'epoch': 2.0}
TEST : {'eval_loss': 0.23903436958789825, 'eval_accuracy': 0.907, 'eval_macro_f1': 0.9069985119761916, 'eval_runtime': 7.0813, 'eval_samples_per_second': 282.435, 'eval_steps_per_second': 8.897, 'epoch': 2.0}


In [36]:
# 8) Quick predictions  (TODO)
# -------------------------------------------------------
# TODO: Implement predict(texts) that returns ["negative"/"positive"] for each text.

# --- YOUR CODE HERE ---
def predict(texts):
    batch = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        logits = trainer.model(**batch).logits
    preds = torch.argmax(logits, axis=1)
    return [id2label[i.item()] for i in preds]

print(predict([
    "Absolutely fantastic movie. I loved every minute!",
    "It was okay, not great, not terrible.",
    "Terrible plot and bad acting."
]))

['positive', 'negative', 'negative']
